In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer

from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    hamming_loss
)

df = pd.read_csv(
    "../data/cleaned/codeforces_cleaned.csv"
)

df["statement"] = df["statement"].fillna("")
df["tags"] = df["tags"].fillna("")

def parse_tags(tags):

    return [
        tag.strip()
        for tag in str(tags).split(";")
        if tag.strip()
    ]


tag_lists = df["tags"].apply(
    parse_tags
)

mlb_topics = MultiLabelBinarizer()

y_topics = mlb_topics.fit_transform(
    tag_lists
)
print(
    "Number of topics:",
    len(mlb_topics.classes_)
)

print(
    mlb_topics.classes_
)

print(
    "Target shape:",
    y_topics.shape
)

Number of topics: 37
['*special' '2-sat' 'binary search' 'bitmasks' 'brute force'
 'chinese remainder theorem' 'combinatorics' 'communication'
 'constructive algorithms' 'data structures' 'dfs and similar'
 'divide and conquer' 'dp' 'dsu' 'expression parsing' 'fft' 'flows'
 'games' 'geometry' 'graph matchings' 'graphs' 'greedy' 'hashing'
 'implementation' 'interactive' 'math' 'matrices' 'meet-in-the-middle'
 'number theory' 'probabilities' 'shortest paths' 'sortings'
 'string suffix structures' 'strings' 'ternary search' 'trees'
 'two pointers']
Target shape: (794, 37)


In [8]:
X_train_text, X_test_text, y_train_topics, y_test_topics = (
    train_test_split(
        df["statement"],
        y_topics,
        test_size=0.20,
        random_state=42
    )
)

In [9]:
topic_tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

X_train_topic = (
    topic_tfidf.fit_transform(
        X_train_text
    )
)

X_test_topic = (
    topic_tfidf.transform(
        X_test_text
    )
)

In [10]:
topic_model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    )
)

topic_model.fit(
    X_train_topic,
    y_train_topics
)

topic_predictions = topic_model.predict(
    X_test_topic
)



In [11]:
micro_f1 = f1_score(
    y_test_topics,
    topic_predictions,
    average="micro",
    zero_division=0
)

macro_f1 = f1_score(
    y_test_topics,
    topic_predictions,
    average="macro",
    zero_division=0
)

precision = precision_score(
    y_test_topics,
    topic_predictions,
    average="micro",
    zero_division=0
)

recall = recall_score(
    y_test_topics,
    topic_predictions,
    average="micro",
    zero_division=0
)

hamming = hamming_loss(
    y_test_topics,
    topic_predictions
)

print("Topic Prediction Baseline")
print("-------------------------")
print("Micro F1:", round(micro_f1, 4))
print("Macro F1:", round(macro_f1, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("Hamming Loss:", round(hamming, 4))

Topic Prediction Baseline
-------------------------
Micro F1: 0.3349
Macro F1: 0.1997
Precision: 0.3819
Recall: 0.2983
Hamming Loss: 0.0938


In [12]:
for i in range(10):

    actual = mlb_topics.inverse_transform(
        y_test_topics[i].reshape(1, -1)
    )[0]

    predicted = mlb_topics.inverse_transform(
        topic_predictions[i].reshape(1, -1)
    )[0]

    print("\nProblem", i + 1)
    print("Actual:   ", actual)
    print("Predicted:", predicted)


Problem 1
Actual:    ('binary search', 'bitmasks', 'brute force', 'constructive algorithms')
Predicted: ('fft', 'math', 'strings')

Problem 2
Actual:    ('brute force', 'constructive algorithms', 'dfs and similar', 'dsu', 'graphs', 'trees')
Predicted: ('constructive algorithms', 'dfs and similar', 'dsu', 'graphs', 'trees')

Problem 3
Actual:    ('constructive algorithms', 'math')
Predicted: ('brute force', 'combinatorics', 'dp', 'math', 'number theory')

Problem 4
Actual:    ('bitmasks', 'dp')
Predicted: ('dp', 'math')

Problem 5
Actual:    ('data structures',)
Predicted: ('bitmasks', 'data structures', 'math')

Problem 6
Actual:    ('brute force', 'constructive algorithms', 'greedy', 'implementation')
Predicted: ('binary search', 'dp')

Problem 7
Actual:    ('binary search', 'geometry')
Predicted: ('geometry',)

Problem 8
Actual:    ('data structures', 'dfs and similar', 'trees')
Predicted: ('data structures', 'dfs and similar', 'dp', 'graphs', 'trees')

Problem 9
Actual:    ('brute 

# Experiment 2 — Tag Distribution and Rare Label Analysis

Before improving the classifier, the distribution of Codeforces tags is
analyzed to identify class imbalance and rare labels.


In [14]:
from collections import Counter

tag_counts = Counter()

for tags in tag_lists:
    for tag in tags:
        tag_counts[tag] += 1

tag_distribution = pd.DataFrame(
    tag_counts.items(),
    columns=["Tag", "Count"]
).sort_values(
    "Count",
    ascending=False
)

tag_distribution

print(tag_distribution.to_string(index=False))
tag_distribution.to_csv(
    "../results/topic_tag_distribution.csv",
    index=False
)


print("Tags with fewer than 5 examples:")

print(
    tag_distribution[
        tag_distribution["Count"] < 5
    ]
)

print("\nTags with fewer than 10 examples:")

print(
    tag_distribution[
        tag_distribution["Count"] < 10
    ]
)

print("\nTags with 20+ examples:")

print(
    tag_distribution[
        tag_distribution["Count"] >= 20
    ]
)




                      Tag  Count
                     math    223
                   greedy    207
                       dp    202
          data structures    179
           implementation    171
  constructive algorithms    150
              brute force    129
            binary search    104
                    trees     99
                   graphs     98
                 sortings     84
          dfs and similar     83
            number theory     67
            combinatorics     59
             two pointers     53
                 bitmasks     52
                  strings     46
       divide and conquer     35
              interactive     35
                 geometry     31
                      dsu     28
            probabilities     21
                    games     21
           shortest paths     20
                      fft     19
                 *special     19
                    flows     17
                  hashing     16
                 matrices      9
          

In [16]:
def parse_tags(tags):

    parsed = []

    for tag in str(tags).split(";"):

        tag = tag.strip()

        if not tag:
            continue

        if tag == "\\*special":
            tag = "*special"

        parsed.append(tag)

    return parsed

tag_lists = df["tags"].apply(parse_tags)

mlb_topics = MultiLabelBinarizer()

y_topics = mlb_topics.fit_transform(
    tag_lists
)

print(mlb_topics.classes_)


['*special' '2-sat' 'binary search' 'bitmasks' 'brute force'
 'chinese remainder theorem' 'combinatorics' 'communication'
 'constructive algorithms' 'data structures' 'dfs and similar'
 'divide and conquer' 'dp' 'dsu' 'expression parsing' 'fft' 'flows'
 'games' 'geometry' 'graph matchings' 'graphs' 'greedy' 'hashing'
 'implementation' 'interactive' 'math' 'matrices' 'meet-in-the-middle'
 'number theory' 'probabilities' 'shortest paths' 'sortings'
 'string suffix structures' 'strings' 'ternary search' 'trees'
 'two pointers']


In [18]:
X_temp, X_test, y_temp, y_test = train_test_split(
    df["statement"],
    y_topics,
    test_size=0.15,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.1765,
    random_state=42
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))


Training: 555
Validation: 119
Testing: 120


In [19]:
topic_tfidf_v2 = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

X_train_vec = topic_tfidf_v2.fit_transform(
    X_train
)

X_val_vec = topic_tfidf_v2.transform(
    X_val
)

X_test_vec = topic_tfidf_v2.transform(
    X_test
)



In [20]:
topic_model_v2 = OneVsRestClassifier(
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    )
)

topic_model_v2.fit(
    X_train_vec,
    y_train
)

val_probabilities = topic_model_v2.predict_proba(
    X_val_vec
)



In [22]:
threshold_results = []

thresholds = [
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50
]

for threshold in thresholds:

    predictions = (
        val_probabilities >= threshold
    ).astype(int)

    micro = f1_score(
        y_val,
        predictions,
        average="micro",
        zero_division=0
    )

    macro = f1_score(
        y_val,
        predictions,
        average="macro",
        zero_division=0
    )

    precision = precision_score(
        y_val,
        predictions,
        average="micro",
        zero_division=0
    )

    recall = recall_score(
        y_val,
        predictions,
        average="micro",
        zero_division=0
    )

    threshold_results.append({
        "Threshold": threshold,
        "Micro_F1": micro,
        "Macro_F1": macro,
        "Precision": precision,
        "Recall": recall
    })

    threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df


threshold_df.sort_values(
    "Micro_F1",
    ascending=False
)


,Threshold,Micro_F1,Macro_F1,Precision,Recall
5,0.45,0.396442,0.245226,0.344371,0.467066
4,0.40,0.384543,0.249467,0.280605,0.610778
6,0.50,0.369384,0.211086,0.415730,0.332335
3,0.35,0.362709,0.242472,0.239654,0.745509
2,0.30,0.336527,0.229003,0.210329,0.841317
1,0.25,0.302209,0.202915,0.181544,0.901198
0,0.20,0.252920,0.170728,0.146114,0.940120


In [23]:
best_threshold_row = (
    threshold_df
    .sort_values(
        "Micro_F1",
        ascending=False
    )
    .iloc[0]
)

best_threshold = (
    best_threshold_row["Threshold"]
)

print(
    "Best threshold:",
    best_threshold
)

print(
    "Validation Micro F1:",
    round(
        best_threshold_row["Micro_F1"],
        4
    )
)

Best threshold: 0.45
Validation Micro F1: 0.3964


In [24]:
test_probabilities = (
    topic_model_v2.predict_proba(
        X_test_vec
    )
)

final_topic_predictions = (
    test_probabilities >= best_threshold
).astype(int)


final_micro_f1 = f1_score(
    y_test,
    final_topic_predictions,
    average="micro",
    zero_division=0
)

final_macro_f1 = f1_score(
    y_test,
    final_topic_predictions,
    average="macro",
    zero_division=0
)

final_precision = precision_score(
    y_test,
    final_topic_predictions,
    average="micro",
    zero_division=0
)

final_recall = recall_score(
    y_test,
    final_topic_predictions,
    average="micro",
    zero_division=0
)

final_hamming = hamming_loss(
    y_test,
    final_topic_predictions
)

print("Threshold Tuned Topic Model")
print("---------------------------")

print(
    "Threshold:",
    best_threshold
)

print(
    "Micro F1:",
    round(final_micro_f1, 4)
)

print(
    "Macro F1:",
    round(final_macro_f1, 4)
)

print(
    "Precision:",
    round(final_precision, 4)
)

print(
    "Recall:",
    round(final_recall, 4)
)

print(
    "Hamming Loss:",
    round(final_hamming, 4)
)


Threshold Tuned Topic Model
---------------------------
Threshold: 0.45
Micro F1: 0.3696
Macro F1: 0.2126
Precision: 0.3192
Recall: 0.4389
Hamming Loss: 0.1214


In [26]:
topic_threshold_results = pd.DataFrame({
    "Metric": [
        "Threshold",
        "Micro F1",
        "Macro F1",
        "Precision",
        "Recall",
        "Hamming Loss"
    ],

    "Value": [
        best_threshold,
        final_micro_f1,
        final_macro_f1,
        final_precision,
        final_recall,
        final_hamming
    ]
})

topic_threshold_results.to_csv(
    "../results/topic_experiment_02_threshold_tuning.csv",
    index=False
)

threshold_df.to_csv(
    "../results/topic_threshold_comparison.csv",
    index=False
)

In [27]:
actual_tags_per_problem = (
    y_test.sum(axis=1).mean()
)

predicted_tags_per_problem = (
    final_topic_predictions
    .sum(axis=1)
    .mean()
)

print(
    "Average actual tags:",
    round(actual_tags_per_problem, 2)
)

print(
    "Average predicted tags:",
    round(predicted_tags_per_problem, 2)
)

Average actual tags: 3.0
Average predicted tags: 4.12


# Experiment 4 — LinearSVC Topic Classification

Linear Support Vector Classification is compared with Logistic Regression
using the same TF-IDF features and primary topic labels.

The purpose is to determine whether a margin-based classifier performs
better for Codeforces topic prediction.

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier

svc_topic_model = OneVsRestClassifier(
    LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=42
    )
)
svc_topic_model.fit(
    X_train_vec,
    y_train
)

svc_val_scores = (
    svc_topic_model.decision_function(
        X_val_vec
    )
)

svc_thresholds = [
    -0.50,
    -0.40,
    -0.30,
    -0.20,
    -0.10,
     0.00,
     0.10,
     0.20
]




In [31]:
svc_threshold_results = []

for threshold in svc_thresholds:

    predictions = (
        svc_val_scores >= threshold
    ).astype(int)

    micro = f1_score(
        y_val,
        predictions,
        average="micro",
        zero_division=0
    )

    macro = f1_score(
        y_val,
        predictions,
        average="macro",
        zero_division=0
    )

    precision = precision_score(
        y_val,
        predictions,
        average="micro",
        zero_division=0
    )

    recall = recall_score(
        y_val,
        predictions,
        average="micro",
        zero_division=0
    )

    svc_threshold_results.append({
        "Threshold": threshold,
        "Micro_F1": micro,
        "Macro_F1": macro,
        "Precision": precision,
        "Recall": recall
    })

    svc_threshold_df = pd.DataFrame(
    svc_threshold_results
)

print(
    svc_threshold_df
    .sort_values(
        "Micro_F1",
        ascending=False
    )
    .to_string(index=False)
)

 Threshold  Micro_F1  Macro_F1  Precision   Recall
      -0.3  0.397546  0.230587   0.336798 0.485030
      -0.2  0.387755  0.227133   0.377841 0.398204
      -0.1  0.377483  0.220714   0.422222 0.341317
      -0.4  0.374343  0.217812   0.288493 0.532934
      -0.5  0.358524  0.206123   0.253731 0.610778
       0.0  0.345588  0.194906   0.447619 0.281437
       0.1  0.307692  0.171992   0.503401 0.221557
       0.2  0.268519  0.154623   0.591837 0.173653


In [32]:
best_svc_row = (
    svc_threshold_df
    .sort_values(
        "Micro_F1",
        ascending=False
    )
    .iloc[0]
)

best_svc_threshold = (
    best_svc_row["Threshold"]
)

print(
    "Best SVC Threshold:",
    best_svc_threshold
)

print(
    "Validation Micro F1:",
    round(
        best_svc_row["Micro_F1"],
        4
    )
)

Best SVC Threshold: -0.3
Validation Micro F1: 0.3975


In [33]:
svc_test_scores = (
    svc_topic_model.decision_function(
        X_test_vec
    )
)

svc_test_predictions = (
    svc_test_scores >= best_svc_threshold
).astype(int)

In [35]:
svc_micro = f1_score(
    y_test,
    svc_test_predictions,
    average="micro",
    zero_division=0
)

svc_macro = f1_score(
    y_test,
    svc_test_predictions,
    average="macro",
    zero_division=0
)

svc_precision = precision_score(
    y_test,
    svc_test_predictions,
    average="micro",
    zero_division=0
)

svc_recall = recall_score(
    y_test,
    svc_test_predictions,
    average="micro",
    zero_division=0
)

svc_hamming = hamming_loss(
    y_test,
    svc_test_predictions
)

print("\nLinearSVC Topic Model")
print("-------------------------")

print("Threshold:", best_svc_threshold)
print("Micro F1:", round(svc_micro, 4))
print("Macro F1:", round(svc_macro, 4))
print("Precision:", round(svc_precision, 4))
print("Recall:", round(svc_recall, 4))
print("Hamming Loss:", round(svc_hamming, 4))

print(
    "Average actual tags:",
    round(
        y_test.sum(axis=1).mean(),
        2
    )
)

print(
    "Average predicted tags:",
    round(
        svc_test_predictions
        .sum(axis=1)
        .mean(),
        2
    )
)


LinearSVC Topic Model
-------------------------
Threshold: -0.3
Micro F1: 0.3582
Macro F1: 0.1996
Precision: 0.3053
Recall: 0.4333
Hamming Loss: 0.1259
Average actual tags: 3.0
Average predicted tags: 4.26


In [41]:
topic_model_comparison = pd.DataFrame({

    "Model": [
        "Logistic Regression",
        "LinearSVC"
    ],

    "Threshold": [
        0.45,
        best_svc_threshold
    ],

    "Micro_F1": [
        0.3696,
        svc_micro
    ],

    "Macro_F1": [
        0.2126,
        svc_macro
    ],

    "Precision": [
        0.3192,
        svc_precision
    ],

    "Recall": [
        0.4389,
        svc_recall
    ],

    "Hamming_Loss": [
        0.1214,
        svc_hamming
    ]
})

topic_model_comparison


topic_model_comparison.to_csv(
    "../results/topic_model_comparison.csv",
    index=False
)



In [ ]:
import json
import joblib

joblib.dump(
    topic_model_v2,
    "../models/topic_model.pkl"
)

joblib.dump(
    topic_tfidf_v2,
    "../models/topic_tfidf_vectorizer.pkl"
)

joblib.dump(
    mlb_topics,
    "../models/topic_label_encoder.pkl"
)

with open(
    "../models/topic_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump({
        "threshold": float(best_threshold),
        "model": "OneVsRestClassifier(LogisticRegression)",
        "num_topics": len(mlb_topics.classes_),
        "topics": list(mlb_topics.classes_)
    }, f, indent=2)

print("Topic prediction models saved successfully.")
